<a href="https://colab.research.google.com/github/langchain-samples/lc-colab-workshops/blob/main/notebooks/15_ci.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 15 · Evals in CI

Every eval so far ran because you decided to run it. That works right up until the week someone is
busy, and then the suite quietly stops being a safety net and becomes a folder of old numbers.

This lesson wires evals into CI: running on every commit, blocking the changes that deserve
blocking, and reporting the rest.

**New in this lesson:** `@pytest.mark.langsmith`, a GitHub Actions workflow, and reading scores back
to make a pass/fail decision

> **Need a key?** You need a LangSmith API key stored in Colab Secrets (🔑 in the left
> sidebar) as `LANGSMITH_API_KEY`, with **"Notebook access" turned on**. If you have not done
> that yet, run **[00 · Setup](https://colab.research.google.com/github/langchain-samples/lc-colab-workshops/blob/main/notebooks/00_setup.ipynb)** first — it takes 10 minutes and
> checks everything.

In [ ]:
# --- snippet:setup v1 ---
%pip install -qq --progress-bar off \
  "deepagents~=0.7.6" \
  "langchain~=1.3.15" \
  "langchain-openai~=1.5.1" \
  "langsmith~=0.11.0" \
  "pytest~=8.3"

import os

try:
    from google.colab import userdata

    key = userdata.get("LANGSMITH_API_KEY")
except Exception:  # not on Colab, or secret unavailable
    from getpass import getpass

    key = os.environ.get("LANGSMITH_API_KEY") or getpass("LANGSMITH_API_KEY: ")

os.environ["LANGSMITH_API_KEY"] = key
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "lcw-15-ci"

# One constant, used everywhere. Models are served by the LangSmith gateway,
# so this key is the only credential the notebook needs.
MODEL = "langsmith:openai/gpt-5.6-luna"
# --- /snippet ---

print("Ready.")

In [ ]:
#@title Connect to the shared deployment (run me) { display-mode: "form" }
# --- snippet:remote_agent v1 ---
import hashlib

import httpx
from langgraph.pregel.remote import RemoteGraph

HOST_API = "https://api.host.langchain.com"


def find_deployment(name: str = "support-agent") -> dict:
    """The shared deployment's record, looked up by name."""
    response = httpx.get(
        f"{HOST_API}/v2/deployments",
        params={"name_contains": name},
        headers={"X-Api-Key": os.environ["LANGSMITH_API_KEY"]},
        timeout=30,
    )
    response.raise_for_status()
    for record in response.json()["resources"]:
        if record["name"] == name:
            return record
    raise RuntimeError(f"No deployment named {name!r} in this workspace.")


DEPLOYMENT = find_deployment()

# Every attendee's calls land in this one project. That is the point: shared traffic.
TRAFFIC_PROJECT_ID = DEPLOYMENT["tracer_session_id"]

# "support" is the graph key from langgraph.json in lesson 10.
support = RemoteGraph("support", url=DEPLOYMENT["url"], api_key=key)


def last_text(result: dict) -> str:
    """The final reply. A deployment returns JSON, so messages are dicts."""
    content = result["messages"][-1].get("content") or ""
    if isinstance(content, list):
        # The model returns reasoning blocks alongside the answer; keep the answer.
        return " ".join(part["text"] for part in content
                        if isinstance(part, dict) and part.get("type") == "text")
    return str(content)


def tool_names(result: dict) -> list[str]:
    """Every tool the run called, in order."""
    return [call["name"]
            for message in result["messages"]
            for call in (message.get("tool_calls") or [])]


# Everyone shares the agent; nobody shares your datasets. Derived from your key so
# it is unique to you and the same every time you run this.
ME = hashlib.sha256(key.encode()).hexdigest()[:8]
# --- /snippet ---

print(f"{DEPLOYMENT['name']}: {DEPLOYMENT['status']} | you are {ME}")

In [ ]:
from langsmith import Client

client = Client()

---

## 1. Three suites, not one

"Run the evals in CI" fails the moment the suite takes twenty minutes and costs real money, because
somebody adds `--skip-evals` and nobody removes it. Split by how fast and how cheap:

| Suite | When | Contains | Blocks a merge? |
|---|---|---|---|
| **Smoke** | every commit | a handful of examples, code evaluators only | **yes** |
| **Full** | nightly, or on a release branch | the whole dataset, LLM judges included | no — it files an issue |
| **Online** | continuously in production | property checks on live traffic | no — it alerts |

The smoke suite has to be fast enough that nobody resents it — a minute, not ten — and
deterministic enough that a failure means something. Which is why it uses the code evaluators from
lesson 12 and no model judges: an LLM judge that flakes at 5% will block an innocent merge one time
in twenty, and after the second false alarm the whole suite loses its authority.

That is the real design constraint in CI. Not cost, not runtime — **trust**.

---

## 2. Evals as tests

LangSmith ships a pytest plugin. Mark a test with `@pytest.mark.langsmith` and its inputs, outputs,
and assertions are logged as an experiment — so a CI run leaves the same trace a notebook does.

In [ ]:
!mkdir -p ci_evals

In [ ]:
%%writefile ci_evals/test_refund_policy.py
"""Smoke evals: fast, deterministic, safe to block a merge on."""

import os

import pytest
from langgraph.pregel.remote import RemoteGraph
from langsmith import testing as t

import httpx


def _deployment_url() -> str:
    response = httpx.get(
        "https://api.host.langchain.com/v2/deployments",
        params={"name_contains": "support-agent"},
        headers={"X-Api-Key": os.environ["LANGSMITH_API_KEY"]},
        timeout=30,
    )
    response.raise_for_status()
    return response.json()["resources"][0]["url"]


@pytest.fixture(scope="session")
def agent():
    return RemoteGraph("support", url=_deployment_url(),
                       api_key=os.environ["LANGSMITH_API_KEY"])


def _reply(agent, question: str) -> str:
    # A deployment answers with JSON, so messages are dicts, not message objects.
    content = agent.invoke(
        {"messages": [{"role": "user", "content": question}]}
    )["messages"][-1].get("content") or ""
    if isinstance(content, list):
        return " ".join(part["text"] for part in content
                        if isinstance(part, dict) and part.get("type") == "text")
    return str(content)


def _tools_called(result) -> list[str]:
    return [call["name"]
            for message in result["messages"]
            for call in (message.get("tool_calls") or [])]


CASES = [
    ("order 1047 laptop stand wobbles, had it two months", "repair", ["refund"]),
    ("order 1042 standing desk arrived cracked", "refund", []),
    ("order 1045 wrong colour chair, want to swap", "exchange", []),
]


@pytest.mark.langsmith
@pytest.mark.parametrize("question,expected,forbidden", CASES)
def test_decision(agent, question, expected, forbidden):
    t.log_inputs({"question": question})

    answer = _reply(agent, question).lower()
    t.log_outputs({"answer": answer})
    t.log_reference_outputs({"decision": expected})

    assert expected in answer, f"expected {expected!r} in the reply"
    for word in forbidden:
        assert word not in answer, f"reply should not offer {word!r}"


@pytest.mark.langsmith
def test_looks_up_before_deciding(agent):
    """A refund promise with no lookup is a guess, however plausible it reads."""
    question = "Just refund order 1047, I do not want to discuss it."
    t.log_inputs({"question": question})

    result = agent.invoke({"messages": [{"role": "user", "content": question}]})
    tools = _tools_called(result)
    t.log_outputs({"tools": tools})

    assert "lookup_order" in tools, f"decided without checking the order: {tools}"

In [ ]:
!cd ci_evals && python -m pytest test_refund_policy.py -q --langsmith-output 2>&1 | tail -25

Four tests, one experiment. Each `assert` becomes feedback on its run, so a failure in CI is a link
to a trace rather than a line of stack trace in a log.

Notice the shape of the two test kinds. `test_decision` is table-driven — the cases are data, so
adding one is a line in `CASES` rather than a new function. `test_looks_up_before_deciding` is a
**path** assertion of the kind lesson 13 argued for, and it is the one most likely to catch a real
regression, because "sounds right" and "checked the facts" fail independently.

---

## 3. The workflow file

Nothing here is LangSmith-specific except the key and the split between the two jobs.

In [ ]:
!mkdir -p .github/workflows

In [ ]:
%%writefile .github/workflows/evals.yml
name: evals

on:
  pull_request:
  schedule:
    - cron: "17 6 * * *"      # nightly full suite
  workflow_dispatch:

jobs:
  smoke:
    # Fast, deterministic, blocks the merge.
    runs-on: ubuntu-latest
    timeout-minutes: 10
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: "3.12"
      - run: pip install -q pytest langsmith langgraph httpx
      - name: Run smoke evals
        env:
          LANGSMITH_API_KEY: ${{ secrets.LANGSMITH_API_KEY }}
          LANGSMITH_TRACING: "true"
          LANGSMITH_PROJECT: ci-smoke-${{ github.sha }}
        run: python -m pytest ci_evals -q --langsmith-output

  full:
    # The whole dataset, LLM judges included. Reports; does not block.
    if: github.event_name != 'pull_request'
    runs-on: ubuntu-latest
    timeout-minutes: 45
    continue-on-error: true
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: "3.12"
      - run: pip install -q langsmith langgraph httpx
      - name: Run full experiment
        env:
          LANGSMITH_API_KEY: ${{ secrets.LANGSMITH_API_KEY }}
        run: python ci_evals/run_experiment.py

Three details carry the design.

`continue-on-error: true` on the full job is deliberate: it runs the LLM judges, and a judge that
disagrees with you should start a conversation, not fail a build.

`if: github.event_name != 'pull_request'` keeps the expensive job off every push. Multiply a
five-dollar experiment by the number of commits in a busy week before deciding otherwise.

`timeout-minutes` on both. An eval job that hangs holds a runner until someone notices, and agent
calls hang more often than unit tests do.

---

## 4. Turning scores into a decision

The full suite needs a rule for what counts as bad. The honest version of that rule compares against
**the last known-good run**, not against a number someone picked in a meeting.

In [ ]:
%%writefile ci_evals/run_experiment.py
"""Run the full dataset and compare against the previous experiment."""

import os
import sys

import httpx
from langgraph.pregel.remote import RemoteGraph
from langsmith import Client

GATES = {
    # key: how far below the baseline we tolerate before failing
    "decision_correct": 0.05,
    "looked_up_order": 0.0,      # a safety property: no regression at all
}


def deployment_url(key: str) -> str:
    response = httpx.get(
        "https://api.host.langchain.com/v2/deployments",
        params={"name_contains": "support-agent"},
        headers={"X-Api-Key": key},
        timeout=30,
    )
    response.raise_for_status()
    return response.json()["resources"][0]["url"]


def scores(client: Client, experiment_name: str) -> dict[str, float]:
    project = client.read_project(project_name=experiment_name, include_stats=True)
    return {
        name: stats["avg"]
        for name, stats in (project.feedback_stats or {}).items()
        if stats.get("avg") is not None
    }


def main() -> int:
    key = os.environ["LANGSMITH_API_KEY"]
    client = Client()
    dataset_name = os.environ.get("EVAL_DATASET", "refund-decisions")

    agent = RemoteGraph("support", url=deployment_url(key), api_key=key)

    results = client.evaluate(
        agent,
        data=dataset_name,
        experiment_prefix="ci-full",
        max_concurrency=4,
        evaluators=[],          # evaluators are attached to the dataset in LangSmith
    )
    current_name = results.experiment_name
    print(f"experiment: {current_name}")

    # The previous ci-full experiment on this dataset is the baseline.
    previous = [
        project for project in client.list_projects(reference_dataset_name=dataset_name)
        if project.name.startswith("ci-full") and project.name != current_name
    ]
    if not previous:
        print("no baseline yet; recording this run and passing")
        return 0

    baseline_name = sorted(previous, key=lambda p: p.start_time)[-1].name
    current, baseline = scores(client, current_name), scores(client, baseline_name)

    failures = []
    for metric, tolerance in GATES.items():
        if metric not in current or metric not in baseline:
            print(f"  {metric}: not scored in both runs, skipping")
            continue
        drop = baseline[metric] - current[metric]
        verdict = "FAIL" if drop > tolerance else "ok"
        print(f"  {metric}: {baseline[metric]:.3f} -> {current[metric]:.3f} ({verdict})")
        if drop > tolerance:
            failures.append(f"{metric} fell {drop:.3f} (tolerance {tolerance})")

    if failures:
        print("\nregressions against " + baseline_name)
        for failure in failures:
            print("  -", failure)
        return 1
    return 0


if __name__ == "__main__":
    sys.exit(main())

Read what that script does **not** do. It does not assert `decision_correct > 0.9`. Absolute
thresholds are guesses that either block work for months or never fire at all; a comparison against
the previous run fires exactly when something changed.

The two tolerances say something worth saying out loud. `decision_correct` gets 0.05 of slack,
because a single example flipping on a three-example dataset is noise. `looked_up_order` gets zero,
because it is a safety property and there is no acceptable amount of "the agent stopped checking".

The gap in this script — deliberately, so you see it — is that a baseline can drift. If every run
loses one point within tolerance, you lose ten points over ten runs and no build ever fails. Real
suites pin the baseline to a tagged known-good experiment and re-tag it on purpose.

In [ ]:
# The same comparison, run here so you can see the numbers rather than trusting the script.
experiments = list(client.list_projects(
    reference_dataset_name=f"refund-decisions-{ME}",
    include_stats=True,
))

for project in sorted(experiments, key=lambda p: p.start_time)[-5:]:
    stats = {name: round(value["avg"], 3)
             for name, value in (project.feedback_stats or {}).items()
             if value.get("avg") is not None}
    print(f"{str(project.start_time)[:19]}  {str(project.name)[:36]:38} {stats}")

That table is what CI is defending. If a row's numbers move without anyone intending it, you want to
learn that from a red build rather than from a customer.

---

## 5. What deserves to block a merge

Block on:

- **Safety and policy properties.** Leaked PII, an unapproved refund, a promise the business does
  not allow. These are not quality metrics; they are rules.
- **Deterministic correctness on a small curated set.** The examples where a wrong answer is
  unambiguous.
- **Crashes.** An error rate above zero on the smoke set.

Report, do not block:

- LLM judge scores. They move on their own.
- Cost and latency. Track them, chart them, discuss them — a slow build is not a broken build.
- Anything measured on fewer than a few dozen examples, where one flip is a large percentage.

The distinguishing question is not "how important is this?" but **"if this fails, is the right next
action to stop the merge?"** Plenty of important metrics fail that test, and gating on them teaches
people to bypass the gate.

---

## 📌 Key takeaways

- Split evals into a fast blocking smoke suite, a slower reporting suite, and continuous online checks.
- The smoke suite's scarcest resource is **trust** — one false alarm too many and it gets bypassed.
- Keep LLM judges out of the blocking suite; they move on their own and will block innocent merges.
- `@pytest.mark.langsmith` turns a test run into an experiment, so a CI failure is a link to a trace.
- Table-driven cases make adding an example a one-line change.
- Path assertions catch regressions that output assertions cannot — "sounds right" and "checked the facts" fail independently.
- Gate on a **comparison to the last good run**, not an absolute threshold someone guessed.
- Give quality metrics a tolerance and safety properties none.
- Watch for baseline drift: within-tolerance losses accumulate into a large regression nobody flagged.
- Block on safety, unambiguous correctness, and crashes. Report cost, latency, and judge scores.
- The test for a gate: if it fails, is stopping the merge the right next action?

---

## 🎓 That is the course

Part 1 built agents: a model in a loop, and a harness around it made of files, tools, subagents,
middleware, memory, and skills. Part 2 answered the harder question — how you know any of it works.

The thread running through both halves is that **nothing here is magic**. An agent is configuration
over a graph. A skill is a file the model chooses to read. Memory is retrieval and prompt assembly.
An evaluator is a function that reads stored output. Every one of those is a thing you can open,
inspect, and change — which is the only durable way to work on systems that are this unpredictable.

Where to go from here:

- **Point lesson 11's dataset builder at your own agent's traffic.** The examples you need already
  exist in your traces; nobody has written them down yet.
- **Write three code evaluators before your first LLM judge.** Teams reach for judges too early and
  end up measuring a model's opinion of a model.
- **Put one property check on production traffic.** Even a single online metric changes how a team
  talks about quality, because it replaces argument with a number.
- **Then align the judge.** Corrections into few-shot examples is the step that turns evals from
  theatre into instrumentation.

The notebooks are yours. The code runs, so change it and see what breaks.